In [6]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


**DUMMY MODEL**

In [7]:
# Predicting any 3 options for all questions
test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
submission = pd.DataFrame({
    'ID': test['id'],
    'Prediction': 'A B C'
})
submission.to_csv('submission.csv', index=False)


# # Predicting the 3 options which have the highest frequency of occurence for all the questions

# train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
# test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

# # Find top 3 most frequent answers
# top3_answers = train['answer'].value_counts().index[:3].tolist()
# prediction = ' '.join(top3_answers)
# print(f"Top 3 answers: {prediction}")

# submission = pd.DataFrame({
#     'ID': test['id'],
#     'Prediction': prediction
# })
# submission.to_csv('submission.csv', index=False)


# # Predicting first option as mode and any other 2 options as the next 2 for all questions

# train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
# test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

# # Find the most common answer in training data
# mode_answer = train['answer'].mode()[0]

# # Create all 5 options and put mode first
# options = ['A', 'B', 'C', 'D', 'E']
# options.remove(mode_answer)
# top3 = mode_answer + ' ' + options[0] + ' ' + options[1]

# submission = pd.DataFrame({
#     'ID': test['id'],
#     'Prediction': top3
# })
# submission.to_csv('submission.csv', index=False)
# print(f"Mode answer: {mode_answer}, Prediction: {top3}")

# Understanding the dataset

In [8]:
import pandas as pd

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("\nFirst few rows:")
print(train.head())
print("\nAnswer distribution:")
print(train['answer'].value_counts())
print("\nSample question:")
print(train['prompt'][0])
print(train[['A','B','C','D','E']].iloc[0])

Train shape: (2000, 8)
Test shape: (500, 7)

First few rows:
   id                                             prompt  \
0   1  Pick the best possible answer: What is Martin ...   
1   2        What is accelerator-based light-ion fusion?   
2   3  Determine the correct option: What is the term...   
3   4  Select the most accurate option: What is Marti...   
4   5  Identify the correct statement: What is the co...   

                                                   A  \
0  Martin Heidegger believes that humans exist wi...   
1  Accelerator-based light-ion fusion is a techni...   
2                                       Blueshifting   
3  Martin Heidegger believes that humans exist wi...   
4  Simultaneity is relative, meaning that two eve...   

                                                   B  \
0  Martin Heidegger believes that humans do not e...   
1  Accelerator-based light-ion fusion is a techni...   
2                                        Redshifting   
3  Martin Heidegg

Key observations:

* Questions are text-heavy (long sentences as options, not just single words)
* Answer distribution: B is most common (490), E is least (324)
* This is a semantic understanding problem — you need to understand meaning, not just keywords
* Thus, a simple TF-IDF model won't work well because the options are very similar in wording.**

**Determining how long the texts are, which determines what model architecture and tokenizer settings to use**

In [9]:
# Check average length of text
import pandas as pd
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

print("Avg prompt length (words):", train['prompt'].str.split().str.len().mean())
print("Avg option A length (words):", train['A'].str.split().str.len().mean())
print("Max prompt length:", train['prompt'].str.split().str.len().max())

Avg prompt length (words): 18.1465
Avg option A length (words): 26.146
Max prompt length: 51


* The texts are manageable in length (avg ~18 words for prompt, ~26 for options, max 51).
* This means we can use max_length=128 or 256 for tokenization.

# Model 1


* Combine prompt + each option into one string
* Convert to numerical features using a simple embedding layer
* Pass through a feedforward neural network
* Output which option is most likely correct


In [10]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import wandb

# better store api 
import os
os.environ["WANDB_API_KEY"] = "your_key_here"

# ---- Load Data ----
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

# ---- Build Vocabulary ----
def tokenize(text):
    return str(text).lower().split()

all_text = []
for col in ['prompt', 'A', 'B', 'C', 'D', 'E']:
    train[col].fillna('').apply(lambda x: all_text.extend(tokenize(x)))
    test[col].fillna('').apply(lambda x: all_text.extend(tokenize(x)))

vocab = {'<PAD>': 0, '<UNK>': 1}
for word, count in Counter(all_text).items():
    if count >= 2:
        vocab[word] = len(vocab)

print(f"Vocab size: {len(vocab)}")

# ---- Encode Text ----
def encode(text, max_len=64):
    tokens = tokenize(text)[:max_len]
    ids = [vocab.get(t, 1) for t in tokens]
    ids += [0] * (max_len - len(ids))
    return ids

label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}

# ---- Dataset ----
class MCQDataset(Dataset):
    def __init__(self, df, is_test=False):
        self.df = df
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt = encode(row['prompt'])
        options = [encode(row[opt]) for opt in ['A','B','C','D','E']]
        # Combine prompt with each option
        combined = [prompt + opt for opt in options]
        x = torch.tensor(combined, dtype=torch.long)  # shape: (5, 128)
        if not self.is_test:
            y = torch.tensor(label_map[row['answer']], dtype=torch.long)
            return x, y
        return x

# ---- Model ----
class MCQModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128, max_len=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.fc = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),  # fixed: embed_dim not embed_dim*128
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        # x: (batch, 5, 128)
        batch_size = x.shape[0]
        x = self.embedding(x)           # (batch, 5, 128, embed_dim)
        x = x.mean(dim=2)               # (batch, 5, embed_dim)
        logits = self.fc(x).squeeze(-1) # (batch, 5)
        return logits

# ---- Training ----
wandb.init(project="24f3002284-t22026", name="model1-scratch", config={
    "embed_dim": 64,
    "hidden_dim": 128,
    "epochs": 10,
    "batch_size": 32,
    "lr": 1e-3
})

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(train, test_size=0.2, random_state=42)

train_loader = DataLoader(MCQDataset(train_df), batch_size=32, shuffle=True)
val_loader = DataLoader(MCQDataset(val_df), batch_size=32)

model = MCQModel(len(vocab)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

for epoch in range(10):
    model.train()
    total_loss, correct = 0, 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (logits.argmax(1) == y).sum().item()

    train_acc = correct / len(train_df)

    # Validation
    model.eval()
    val_correct = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            val_correct += (logits.argmax(1) == y).sum().item()
    val_acc = val_correct / len(val_df)

    print(f"Epoch {epoch+1}: Loss={total_loss/len(train_loader):.4f}, Train Acc={train_acc:.4f}, Val Acc={val_acc:.4f}")
    wandb.log({"epoch": epoch+1, "loss": total_loss/len(train_loader),
               "train_acc": train_acc, "val_acc": val_acc})

# Save model
torch.save(model.state_dict(), 'model1_scratch.pth')
wandb.save('model1_scratch.pth')
wandb.finish()
print("Training done!")

Vocab size: 3816


Using device: cuda
Epoch 1: Loss=1.6068, Train Acc=0.2300, Val Acc=0.3850
Epoch 2: Loss=1.5924, Train Acc=0.3394, Val Acc=0.4525
Epoch 3: Loss=1.5280, Train Acc=0.4575, Val Acc=0.5650
Epoch 4: Loss=1.2974, Train Acc=0.5837, Val Acc=0.6575
Epoch 5: Loss=0.9631, Train Acc=0.7106, Val Acc=0.7600
Epoch 6: Loss=0.7017, Train Acc=0.8000, Val Acc=0.8425
Epoch 7: Loss=0.5271, Train Acc=0.8625, Val Acc=0.8800
Epoch 8: Loss=0.4007, Train Acc=0.9019, Val Acc=0.8900
Epoch 9: Loss=0.3173, Train Acc=0.9150, Val Acc=0.9125


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


Epoch 10: Loss=0.2464, Train Acc=0.9375, Val Acc=0.9300


epoch,▁▂▃▃▄▅▆▆▇█
loss,███▆▅▃▂▂▁▁
train_acc,▁▂▃▅▆▇▇███
val_acc,▁▂▃▅▆▇▇▇██
epoch,10
loss,0.24637
train_acc,0.9375
val_acc,0.93


Training done!


* This high accuracy might be due to the model memorizing patterns.

**Inference**

In [11]:
# Inference with Model 1
model.eval()
test_dataset = MCQDataset(test, is_test=True)
test_loader = DataLoader(test_dataset, batch_size=32)

all_preds = []
with torch.no_grad():
    for x in test_loader:
        x = x.to(device)
        logits = model(x)
        # Get top 3 predictions
        top3 = torch.topk(logits, 3, dim=1).indices.cpu().numpy()
        all_preds.extend(top3)

options = ['A', 'B', 'C', 'D', 'E']
predictions = [' '.join([options[i] for i in pred]) for pred in all_preds]

submission = pd.DataFrame({'ID': test['id'], 'Prediction': predictions})
submission.to_csv('submission.csv', index=False)
print(submission.head())

   ID Prediction
0   1      D E A
1   2      B C D
2   3      B E A
3   4      E A C
4   5      C D A


* THis was a simple embedding model. It won't generalize well to unseen test data even if training accuracy is high.
* The 93% training accuracy implies overfitting.